# OPTIMIZATION 8 — Predictive Multi-Buffer Landsat Imputation

## Root Cause Diagnosis: Why Opt 7 Regressed from 0.3499 → 0.3109

| Problem | Root Cause | Impact |
|---|---|---|
| **Train/Val spectral mismatch** | Training multi-buffer: only 16% real data, filled with **global medians**. Val: 86% real data. Model trained on fake-median spectra, scored on real spectra. | Distribution mismatch → poor generalization |
| **1,085 orig Landsat NaN rows** | `nir` missing for 11.6% of training data → `NDWI`, `green_nir_ratio`, `swir_ratio` all NaN for same rows. Opt 7 median-imputed → corrupted features. | Model A CV dropped: 0.31 (Opt 5) → 0.22 (Opt 7) |
| **Negative Model B** | Only 1,517 rows for Model B (51 features) → severe spatial overfitting. CV R² = −0.031. Blending 15% of Model B into val predictions **actively hurt** | −0.04 penalty on final score |

## Opt 8 Fix — Three-Stage Predictive Imputation

```
Stage 1: Fix 1,085 orig Landsat NaN rows
         KNN imputation using [Lat, Lon, month, TerraClimate] → find K=7 spatial neighbors

Stage 2: Build Spectral Imputer for multi-buffer gaps
         Train ExtraTrees on 1,514 rows WITH real data:
         [Lat, Lon, month, season, TC_8_vars] → [nir/green/swir16/swir22/NDMI/MNDWI × 3 buffers]
         Apply to ALL 9,319 rows — fills 83% missing with realistic predicted values

Stage 3: Engineer ALL derived features AFTER complete imputation
         No NaN propagation | Full feature availability for all 9,319 training rows

Main Model: Proven Opt 15 architecture (ET 65% + RF 35%, leaf=10, 3-seed average)
            Trained on ALL 9,319 rows with ~50 features
```

In [1]:
# ══════════════════════════════════════════════════════════════════════════════
# OPT 8 — CELL 1: DATA LOADING + IMPUTATION PIPELINE
# ══════════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.metrics import r2_score
import time

start = time.time()
print('=' * 70)
print('OPTIMIZATION 8 — Predictive Multi-Buffer Landsat Imputation')
print('=' * 70)

# ── LOAD BASE DATA ────────────────────────────────────────────────────────────
print('\n[1/6] Loading base data...')

train_df = pd.read_csv('water_quality_training_dataset.csv')
train_df['Sample Date'] = pd.to_datetime(train_df['Sample Date'], dayfirst=True)
train_df['YearMonth']   = train_df['Sample Date'].dt.to_period('M').astype(str)
train_df['month']       = train_df['Sample Date'].dt.month
train_df['season']      = ((train_df['month'] % 12) + 3) // 3
train_df['day_of_year'] = train_df['Sample Date'].dt.dayofyear
train_df['Location_ID'] = (train_df['Latitude'].round(4).astype(str)
                           + '_' + train_df['Longitude'].round(4).astype(str))

template_df = pd.read_csv('submission_template.csv')
template_df['Sample Date'] = pd.to_datetime(template_df['Sample Date'], dayfirst=True)
template_df['YearMonth']   = template_df['Sample Date'].dt.to_period('M').astype(str)
template_df['month']       = template_df['Sample Date'].dt.month
template_df['season']      = ((template_df['month'] % 12) + 3) // 3
template_df['day_of_year'] = template_df['Sample Date'].dt.dayofyear

tc_train = pd.read_csv('terraclimate_features_training_final.csv')
tc_val   = pd.read_csv('terraclimate_features_validation_final.csv')
ls_orig_train = pd.read_csv('landsat_features_training.csv')
ls_orig_val   = pd.read_csv('landsat_features_validation.csv')

print(f'   Training labels   : {train_df.shape}')
print(f'   TC train/val      : {tc_train.shape} / {tc_val.shape}')
print(f'   LS orig train/val : {ls_orig_train.shape} / {ls_orig_val.shape}')

# ── ROW-ALIGNED BASE MERGE ────────────────────────────────────────────────────
print('\n[2/6] Building base merged frames (row-aligned concat)...')

def row_concat(label_df, tc_df, ls_df):
    out = label_df.copy().reset_index(drop=True)
    tc  = tc_df.drop(columns=['Latitude','Longitude','Sample Date'], errors='ignore')
    ls  = ls_df.drop(columns=['Latitude','Longitude','Sample Date'], errors='ignore')
    return pd.concat([out, tc.reset_index(drop=True), ls.reset_index(drop=True)], axis=1)

train_base = row_concat(train_df, tc_train, ls_orig_train)
val_base   = row_concat(template_df, tc_val, ls_orig_val)

TC_VARS    = ['pet','ppt','soil','tmax','tmin','aet','def','srad']
ORIG_BANDS = ['nir','green','swir16','swir22','NDMI','MNDWI']

print(f'   train_base : {train_base.shape}')
print(f'   nir NaNs in train_base : {train_base["nir"].isna().sum()} / {len(train_base)}')
print(f'   pet NaNs (TC check)    : {train_base["pet"].isna().sum()}')

# ── STAGE 1: FIX ORIGINAL LANDSAT NaNs WITH SPATIAL-TEMPORAL KNN ─────────────
print('\n[3/6] Stage 1 — Spatial-temporal KNN fix for 1,085 orig Landsat NaNs...')

IMPUTE_FEATS = ['Latitude','Longitude','month'] + TC_VARS  # 11 spatiotemporal features

has_orig   = train_base['nir'].notna()
n_valid    = has_orig.sum()
n_missing  = (~has_orig).sum()
print(f'   Valid orig Landsat rows  : {n_valid}')
print(f'   Missing orig Landsat rows: {n_missing}')

knn_X_train = train_base.loc[has_orig, IMPUTE_FEATS].fillna(train_base[IMPUTE_FEATS].median())
knn_y_train = train_base.loc[has_orig, ORIG_BANDS]

knn_scaler = StandardScaler()
knn_X_scaled = knn_scaler.fit_transform(knn_X_train)

print('   Training KNN imputer (K=7, distance-weighted)...')
knn_imp = KNeighborsRegressor(n_neighbors=7, weights='distance', n_jobs=-1)
knn_imp.fit(knn_X_scaled, knn_y_train)

# Impute missing training rows
miss_feats  = train_base.loc[~has_orig, IMPUTE_FEATS].fillna(train_base[IMPUTE_FEATS].median())
miss_scaled = knn_scaler.transform(miss_feats)
knn_preds   = knn_imp.predict(miss_scaled)

train_base_fixed = train_base.copy()
for j, col in enumerate(ORIG_BANDS):
    train_base_fixed.loc[~has_orig, col] = knn_preds[:, j]

print(f'   ✓ After KNN fix: nir NaNs = {train_base_fixed["nir"].isna().sum()}')

# Fix val original Landsat NaNs if any
val_base_fixed = val_base.copy()
has_orig_val = val_base['nir'].notna()
n_miss_val = (~has_orig_val).sum()
if n_miss_val > 0:
    print(f'   Fixing {n_miss_val} val orig Landsat NaNs...')
    vm_feats  = val_base.loc[~has_orig_val, IMPUTE_FEATS].fillna(train_base[IMPUTE_FEATS].median())
    vm_scaled = knn_scaler.transform(vm_feats)
    vm_preds  = knn_imp.predict(vm_scaled)
    for j, col in enumerate(ORIG_BANDS):
        val_base_fixed.loc[~has_orig_val, col] = vm_preds[:, j]
else:
    print(f'   Val orig Landsat: all rows valid ✓')

# ── LOAD MULTI-BUFFER LANDSAT ─────────────────────────────────────────────────
print('\n[4/6] Loading multi-buffer Landsat files...')

ls_multi_train = {}
for buf in [50, 150, 200]:
    df = pd.read_csv(f'landsat_{buf}m_combined.csv')
    df.columns = df.columns.str.upper().str.strip().str.replace(' ', '_')
    if 'SAMPLE_DATE' in df.columns:
        df['SAMPLE_DATE'] = pd.to_datetime(df['SAMPLE_DATE'], errors='coerce')
        df['YEARMONTH'] = df['SAMPLE_DATE'].dt.to_period('M').astype(str)
    df = df.rename(columns={'LATITUDE':'Latitude','LONGITUDE':'Longitude','YEARMONTH':'YearMonth'})
    for col in ['NIR','GREEN','SWIR16','SWIR22','NDMI','MNDWI']:
        if col in df.columns:
            df = df.rename(columns={col: f'{col.lower()}_{buf}m'})
    keep = ['Latitude','Longitude','YearMonth'] + [c for c in df.columns if f'_{buf}m' in c]
    df = df[keep].drop_duplicates(subset=['Latitude','Longitude','YearMonth'])
    ls_multi_train[buf] = df
    nir = f'nir_{buf}m'
    cov = 100 * df[nir].notna().mean() if nir in df.columns else 0
    print(f'   Train {buf}m: {df.shape}  NIR real coverage: {cov:.1f}%')

ls_multi_val = {}
for buf in [50, 150, 200]:
    df = pd.read_csv(f'landsat_features_{buf}m_combined.csv')
    df.columns = df.columns.str.strip()
    for col in ['nir','green','swir16','swir22','NDMI','MNDWI']:
        if col in df.columns:
            df = df.rename(columns={col: f'{col.lower()}_{buf}m'})
    keep = ['Latitude','Longitude'] + [c for c in df.columns if f'_{buf}m' in c]
    df = df[keep].drop_duplicates(subset=['Latitude','Longitude'])
    ls_multi_val[buf] = df
    nir = f'nir_{buf}m'
    cov = 100 * df[nir].notna().mean() if nir in df.columns else 0
    print(f'   Val   {buf}m: {df.shape}  NIR real coverage: {cov:.1f}%')

# Merge multi-buffer into training/val
print('\n[5/6] Merging multi-buffer into training data...')
train_full = train_base_fixed.copy()
for buf in [50, 150, 200]:
    train_full = train_full.merge(
        ls_multi_train[buf], on=['Latitude','Longitude','YearMonth'], how='left')
    n = train_full[f'nir_{buf}m'].notna().sum()
    print(f'   Train {buf}m real matches: {n:,}/{len(train_full):,} ({100*n/len(train_full):.1f}%)')

val_full = val_base_fixed.copy()
for buf in [50, 150, 200]:
    val_full = val_full.merge(ls_multi_val[buf], on=['Latitude','Longitude'], how='left')
    n = val_full[f'nir_{buf}m'].notna().sum()
    print(f'   Val   {buf}m real matches: {n:,}/{len(val_full):,} ({100*n/len(val_full):.1f}%)')
assert len(val_full) == 200, f'Val exploded: {len(val_full)} rows!'

# ── STAGE 2: TRAIN PREDICTIVE SPECTRAL IMPUTER ───────────────────────────────
# KEY INNOVATION: Instead of median-imputing 83% of multi-buffer rows,
# train an ExtraTrees model to PREDICT spectral values from climate + location.
# Training data: 1,514 rows WITH real multi-buffer measurements.
# Apply to ALL 9,319 rows → coherent spectral distribution for model training.
print('\n[6/6] Stage 2 — Training predictive spectral imputer (ET on 1,514 real rows)...')

IMPUTER_FEATS = ['Latitude','Longitude','month','season'] + TC_VARS  # 14 features, 100% available
BUF_COLS = [f'{b}_{buf}m' for buf in [50,150,200]
             for b in ['nir','green','swir16','swir22','ndmi','mndwi']]
buf_cols_avail = [c for c in BUF_COLS if c in train_full.columns]

has_any_buf = train_full[buf_cols_avail].notna().any(axis=1)
print(f'   Imputer training rows (have any real buffer data): {has_any_buf.sum()}')

imp_X = train_full.loc[has_any_buf, IMPUTER_FEATS].fillna(
    train_full[IMPUTER_FEATS].median())
imp_y = train_full.loc[has_any_buf, buf_cols_avail].copy()
for col in imp_y.columns:
    imp_y[col].fillna(imp_y[col].median(), inplace=True)

imp_scaler = StandardScaler()
imp_X_scaled = imp_scaler.fit_transform(imp_X)

# Quick holdout evaluation before training full imputer
Xsi_tr, Xsi_va, ysi_tr, ysi_va = train_test_split(
    imp_X_scaled, imp_y, test_size=0.2, random_state=42)
quick_imp = ExtraTreesRegressor(n_estimators=100, max_depth=12,
                                 min_samples_leaf=3, random_state=42, n_jobs=-1)
quick_imp.fit(Xsi_tr, ysi_tr)
imp_r2 = r2_score(ysi_va, quick_imp.predict(Xsi_va), multioutput='variance_weighted')
print(f'   Spectral imputer R² (20% holdout): {imp_r2:.4f}')
print(f'   → R² > 0.3 = imputer captures real patterns (better than median fill)')

# Train full imputer on all available data
spectral_imputer = ExtraTreesRegressor(
    n_estimators=200, max_depth=15, min_samples_leaf=3,
    max_features='sqrt', random_state=42, n_jobs=-1
)
spectral_imputer.fit(imp_X_scaled, imp_y)

# Apply to ALL training rows
all_imp_X = train_full[IMPUTER_FEATS].fillna(train_full[IMPUTER_FEATS].median())
all_imp_X_scaled = imp_scaler.transform(all_imp_X)
all_imputed = spectral_imputer.predict(all_imp_X_scaled)
all_imputed_df = pd.DataFrame(all_imputed, columns=buf_cols_avail, index=train_full.index)

# Fill: real data takes priority, imputed fills the gaps
train_imputed = train_full.copy()
for col in buf_cols_avail:
    mask = train_imputed[col].isna()
    train_imputed.loc[mask, col] = all_imputed_df.loc[mask, col]

# Apply same imputer to val missing rows (28/200)
val_imp_X = val_full[IMPUTER_FEATS].fillna(train_full[IMPUTER_FEATS].median())
val_imp_X_scaled = imp_scaler.transform(val_imp_X)
val_imputed_bands = spectral_imputer.predict(val_imp_X_scaled)
val_imputed_df = pd.DataFrame(val_imputed_bands, columns=buf_cols_avail, index=val_full.index)

val_imputed = val_full.copy()
for col in buf_cols_avail:
    mask = val_imputed[col].isna()
    val_imputed.loc[mask, col] = val_imputed_df.loc[mask, col]

print('\n   Coverage after full imputation:')
for col in [f'nir_{buf}m' for buf in [50,150,200] if f'nir_{buf}m' in train_imputed.columns]:
    n = train_imputed[col].notna().sum()
    print(f'   {col}: {n}/{len(train_imputed)} ({100*n/len(train_imputed):.1f}%)')

print(f'\n✅ Cell 1 complete — full imputation pipeline done')
print(f'   Train: {len(train_imputed)} rows | Val: {len(val_imputed)} rows')

OPTIMIZATION 8 — Predictive Multi-Buffer Landsat Imputation

[1/6] Loading base data...
   Training labels   : (9319, 11)
   TC train/val      : (9319, 11) / (200, 11)
   LS orig train/val : (9319, 9) / (200, 9)

[2/6] Building base merged frames (row-aligned concat)...
   train_base : (9319, 25)
   nir NaNs in train_base : 1085 / 9319
   pet NaNs (TC check)    : 0

[3/6] Stage 1 — Spatial-temporal KNN fix for 1,085 orig Landsat NaNs...
   Valid orig Landsat rows  : 8234
   Missing orig Landsat rows: 1085
   Training KNN imputer (K=7, distance-weighted)...
   ✓ After KNN fix: nir NaNs = 0
   Fixing 19 val orig Landsat NaNs...

[4/6] Loading multi-buffer Landsat files...
   Train 50m: (6404, 9)  NIR real coverage: 19.6%
   Train 150m: (6404, 9)  NIR real coverage: 19.4%
   Train 200m: (6404, 9)  NIR real coverage: 19.2%
   Val   50m: (186, 8)  NIR real coverage: 90.3%
   Val   150m: (186, 8)  NIR real coverage: 91.9%
   Val   200m: (186, 8)  NIR real coverage: 91.9%

[5/6] Merging multi

In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# OPT 8 — CELL 2: FEATURE ENGINEERING + MODEL TRAINING + SUBMISSION
# ══════════════════════════════════════════════════════════════════════════════

print('=' * 70)
print('OPT 8 CELL 2: Feature Engineering + Training')
print('=' * 70)

# ── STAGE 3: FEATURE ENGINEERING AFTER COMPLETE IMPUTATION ───────────────────
# Critical: compute derived features AFTER imputation so no NaN propagation.
# Opt 7 computed NDWI from NaN nir → NDWI also NaN → 1,085 corrupted rows.
print('\n[1/4] Stage 3 — Engineering ALL features on fully-imputed data...')

def add_all_features(df):
    df = df.copy()
    eps = 1e-10

    # Original Landsat derived (now NaN-free after Stage 1 KNN fix)
    if 'nir' in df.columns and 'green' in df.columns:
        df['NDWI']            = (df['green'] - df['nir']) / (df['green'] + df['nir'] + eps)
        df['green_nir_ratio'] = df['green'] / (df['nir'] + eps)
    if 'swir22' in df.columns and 'swir16' in df.columns:
        df['swir_ratio']      = df['swir22'] / (df['swir16'] + eps)

    # TerraClimate derived
    if 'tmax' in df.columns and 'tmin' in df.columns:
        df['temp_range']      = df['tmax'] - df['tmin']
    if 'ppt' in df.columns and 'pet' in df.columns:
        df['water_balance']   = df['ppt'] - df['pet']
        df['aridity_index']   = df['pet'] / (df['ppt'] + eps)
    if 'soil' in df.columns and 'ppt' in df.columns:
        df['soil_saturation'] = df['soil'] / (df['ppt'] + eps)

    # Cross-buffer spatial heterogeneity stats (now valid for ALL rows after Stage 2)
    for spec in ['nir','green','swir16','swir22','ndmi','mndwi']:
        cols = [f'{spec}_{b}m' for b in [50,150,200] if f'{spec}_{b}m' in df.columns]
        if len(cols) >= 2:
            df[f'{spec}_buf_mean']  = df[cols].mean(axis=1)
            df[f'{spec}_buf_std']   = df[cols].std(axis=1)   # riparian heterogeneity
            df[f'{spec}_buf_range'] = df[cols].max(axis=1) - df[cols].min(axis=1)

    # Buffer-scale spectral ratios
    for b in [50, 150, 200]:
        n_c, s_c, g_c = f'nir_{b}m', f'swir16_{b}m', f'green_{b}m'
        if n_c in df.columns and s_c in df.columns:
            df[f'nir_swir_{b}m'] = df[n_c] / (df[s_c] + eps)
        if n_c in df.columns and g_c in df.columns:
            df[f'NDWI_{b}m'] = (df[g_c] - df[n_c]) / (df[g_c] + df[n_c] + eps)

    return df

train_feat = add_all_features(train_imputed)
val_feat   = add_all_features(val_imputed)

print(f'   train_feat: {train_feat.shape}')
print(f'   val_feat  : {val_feat.shape}')

# ── FEATURE SELECTION ─────────────────────────────────────────────────────────
print('\n[2/4] Defining feature sets...')

OPT15_CORE = [
    'nir','green','swir16','swir22','NDMI','MNDWI',
    'NDWI','swir_ratio','green_nir_ratio',
    'pet','ppt','soil','tmax','tmin',
    'temp_range','water_balance','aridity_index','soil_saturation',
    'month','season','day_of_year'
]
TC_EXTRA    = ['aet','def','srad']   # not in original Opt15 but proven useful
MULTI_RAW   = [f'{b}_{buf}m' for buf in [50,150,200]
               for b in ['nir','green','swir16','swir22','ndmi','mndwi']]
CROSS_STATS = ([f'{b}_{s}' for b in ['nir','green','swir16','swir22','ndmi','mndwi']
                for s in ['buf_mean','buf_std','buf_range']]
               + [f'nir_swir_{buf}m' for buf in [50,150,200]]
               + [f'NDWI_{buf}m' for buf in [50,150,200]])

ALL_FEATS = OPT15_CORE + TC_EXTRA + MULTI_RAW + CROSS_STATS
seen = set()
avail_feats = [f for f in ALL_FEATS
               if f in train_feat.columns and f in val_feat.columns
               and not (f in seen or seen.add(f))]

print(f'   OPT15 core     : {len([f for f in OPT15_CORE if f in avail_feats])}')
print(f'   TC extra       : {len([f for f in TC_EXTRA if f in avail_feats])}')
print(f'   Multi-buf raw  : {len([f for f in MULTI_RAW if f in avail_feats])}')
print(f'   Cross-buf stats: {len([f for f in CROSS_STATS if f in avail_feats])}')
print(f'   Total          : {len(avail_feats)} features')

TARGETS = ['Total Alkalinity','Electrical Conductance','Dissolved Reactive Phosphorus']
y_train_arr  = train_feat[TARGETS].values
location_ids = train_feat['Location_ID']

X_train_raw = train_feat[avail_feats].copy()
X_val_raw   = val_feat[avail_feats].copy()

# Final safety NaN fill with training medians
train_medians = X_train_raw.median()
X_train_raw.fillna(train_medians, inplace=True)
X_val_raw.fillna(train_medians, inplace=True)

leftover_nan = X_train_raw.isna().sum().sum()
print(f'   Residual NaNs after full pipeline: {leftover_nan}  (should be 0)')

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_val   = scaler.transform(X_val_raw)
print(f'   X_train: {X_train.shape}  |  X_val: {X_val.shape}')

# ── SPATIAL CV ────────────────────────────────────────────────────────────────
print('\n[3/4] Spatial CV (5-fold GroupKFold by location)...')

ET_PARAMS = dict(n_estimators=400, max_depth=20, min_samples_leaf=10,
                 max_features='sqrt', n_jobs=-1)
RF_PARAMS = dict(n_estimators=300, max_depth=18, min_samples_leaf=10,
                 max_features='sqrt', n_jobs=-1)
ET_W, RF_W = 0.65, 0.35
SEEDS = [42, 100, 200]
gkf   = GroupKFold(n_splits=5)

cv_results = {}
for i, tname in enumerate(['TA','EC','DRP']):
    y_t = y_train_arr[:, i]
    blend_folds = []
    et_folds, rf_folds = [], []
    for tr_idx, va_idx in gkf.split(X_train, y_t, groups=location_ids):
        et = ExtraTreesRegressor(**ET_PARAMS, random_state=42)
        rf = RandomForestRegressor(**RF_PARAMS, random_state=42)
        et.fit(X_train[tr_idx], y_t[tr_idx])
        rf.fit(X_train[tr_idx], y_t[tr_idx])
        p_et = et.predict(X_train[va_idx])
        p_rf = rf.predict(X_train[va_idx])
        p_b  = ET_W * p_et + RF_W * p_rf
        et_folds.append(r2_score(y_t[va_idx], p_et))
        rf_folds.append(r2_score(y_t[va_idx], p_rf))
        blend_folds.append(r2_score(y_t[va_idx], p_b))
    cv_results[tname] = {'et': np.mean(et_folds), 'rf': np.mean(rf_folds),
                          'blend': np.mean(blend_folds), 'folds': blend_folds}
    print(f'   {tname}: ET={np.mean(et_folds):.4f}  RF={np.mean(rf_folds):.4f}  '
          f'Blend={np.mean(blend_folds):.4f}  '
          f'{[f"{x:.3f}" for x in blend_folds]}')

overall_cv = np.mean([cv_results[t]['blend'] for t in ['TA','EC','DRP']])
print(f'\n   Overall Blend CV R²  : {overall_cv:.4f}')
print(f'   Reference — Opt 7   : 0.2217')
print(f'   Reference — Opt 2/5 : ~0.31')
if overall_cv > 0.3109:
    print(f'   ✅ BEATS Opt 7 leaderboard score (0.3109)')
elif overall_cv > 0.2217:
    print(f'   ✅ BEATS Opt 7 CV (0.2217) — likely beats leaderboard too')
else:
    print(f'   ⚠ Check feature pipeline')

# ── FULL TRAINING + SUBMISSION ────────────────────────────────────────────────
print('\n[4/4] Training full 3-seed ensemble + saving submission...')

val_preds_all = []
for seed in SEEDS:
    seed_preds = np.zeros((len(X_val), 3))
    for i in range(3):
        y_t = y_train_arr[:, i]
        et = ExtraTreesRegressor(**ET_PARAMS, random_state=seed)
        rf = RandomForestRegressor(**RF_PARAMS, random_state=seed)
        et.fit(X_train, y_t)
        rf.fit(X_train, y_t)
        seed_preds[:, i] = ET_W * et.predict(X_val) + RF_W * rf.predict(X_val)
    val_preds_all.append(seed_preds)
    print(f'   Seed {seed} done')

final_preds = np.clip(np.mean(val_preds_all, axis=0), 0, None)

print('\n   Final prediction statistics:')
for i, (col, tname) in enumerate(zip(TARGETS, ['TA','EC','DRP'])):
    p = final_preds[:, i]
    print(f'     {tname}: min={p.min():.2f}  max={p.max():.2f}  '
          f'mean={p.mean():.2f}  std={p.std():.2f}')

print(f'\n   Row 0 sanity check (Opt 15 ref: TA≈93, EC≈282, DRP≈27):')
print(f'   TA={final_preds[0,0]:.2f}  EC={final_preds[0,1]:.2f}  DRP={final_preds[0,2]:.2f}')

submission = pd.read_csv('submission_template.csv')
assert len(submission) == 200
for i, col in enumerate(TARGETS):
    submission[col] = final_preds[:, i]
assert submission[TARGETS].isna().sum().sum() == 0
submission.to_csv('submission.csv', index=False)

elapsed = time.time() - start
print('\n' + '=' * 70)
print('OPTIMIZATION 8 COMPLETE')
print('=' * 70)
print(f'   Training rows      : {len(X_train):,}  (Opt 7 Model B had 1,517)')
print(f'   Feature count      : {len(avail_feats)}')
print(f'   Overall CV R²      : {overall_cv:.4f}  (Opt 7: 0.2217)')
print(f'   Elapsed            : {elapsed:.0f}s')
print(f'   Saved              : submission.csv')
print(f'')
print(f'   What changed vs Opt 7:')
print(f'   1. KNN spatial fix for 1,085 orig Landsat NaN rows')
print(f'   2. Predictive ET imputer for 83% missing multi-buffer rows')
print(f'   3. Derived features computed AFTER full imputation (no NaN propagation)')
print(f'   4. Single model on ALL 9,319 rows (no negative-CV Model B blend)')
print(submission.head(6).to_string(index=False))

OPT 8 CELL 2: Feature Engineering + Training

[1/4] Stage 3 — Engineering ALL features on fully-imputed data...
   train_feat: (9319, 74)
   val_feat  : (200, 73)

[2/4] Defining feature sets...
   OPT15 core     : 21
   TC extra       : 3
   Multi-buf raw  : 18
   Cross-buf stats: 24
   Total          : 66 features
   Residual NaNs after full pipeline: 0  (should be 0)
   X_train: (9319, 66)  |  X_val: (200, 66)

[3/4] Spatial CV (5-fold GroupKFold by location)...
   TA: ET=0.2830  RF=0.2606  Blend=0.2799  ['0.235', '0.108', '0.323', '0.317', '0.417']
   EC: ET=0.3642  RF=0.3605  Blend=0.3680  ['0.318', '0.334', '0.397', '0.283', '0.507']
   DRP: ET=0.1422  RF=0.1203  Blend=0.1390  ['0.133', '0.157', '0.201', '0.194', '0.010']

   Overall Blend CV R²  : 0.2623
   Reference — Opt 7   : 0.2217
   Reference — Opt 2/5 : ~0.31
   ✅ BEATS Opt 7 CV (0.2217) — likely beats leaderboard too

[4/4] Training full 3-seed ensemble + saving submission...
   Seed 42 done
   Seed 100 done
   Seed 200 

In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# OPTIMIZATION 9 — CELL 1: Feature Engineering (Clean 24-Feature Set)
# ══════════════════════════════════════════════════════════════════════════════
#
# STRATEGY: Keep Opt 8's KNN fix (genuine quality improvement) but DROP
# the 45 synthetic multi-buffer features that compressed predictions.
#
# Carries over from Opt 8 Cell 1 (already in memory):
#   train_base_fixed — 9,319 rows, orig Landsat NaNs fixed by KNN
#   val_base_fixed   — 200 rows,  orig Landsat NaNs fixed by KNN
#
# What we do here:
#   Proven OPT15 features (21) + TC extras aet/def/srad (3) = 24 features
#   Derived features computed on NaN-free data (no NDWI corruption)
#   All 9,319 training rows — NO synthetic multi-buffer bands
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 70)
print("OPTIMIZATION 9 — Clean 24-Feature Set on KNN-Fixed Data")
print("=" * 70)
print("Strategy: Opt 8 KNN fix  +  Proven OPT15 features  -  Synthetic buffer noise")

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
import time

start9 = time.time()

# ── BUILD FEATURES ON KNN-FIXED DATA (no multi-buffer) ───────────────────────
print("\n[1/3] Engineering proven features on KNN-fixed data...")

def add_opt15_features(df):
    """Compute OPT15 derived features — safe because orig Landsat NaNs are fixed."""
    df = df.copy()
    eps = 1e-10

    # Spectral derived (all valid now — KNN filled the 1,085 NaN rows)
    if 'nir' in df.columns and 'green' in df.columns:
        df['NDWI']            = (df['green'] - df['nir']) / (df['green'] + df['nir'] + eps)
        df['green_nir_ratio'] = df['green'] / (df['nir'] + eps)
    if 'swir22' in df.columns and 'swir16' in df.columns:
        df['swir_ratio']      = df['swir22'] / (df['swir16'] + eps)

    # TerraClimate derived
    if 'tmax' in df.columns and 'tmin' in df.columns:
        df['temp_range']      = df['tmax'] - df['tmin']
    if 'ppt' in df.columns and 'pet' in df.columns:
        df['water_balance']   = df['ppt'] - df['pet']
        df['aridity_index']   = df['pet'] / (df['ppt'] + eps)
    if 'soil' in df.columns and 'ppt' in df.columns:
        df['soil_saturation'] = df['soil'] / (df['ppt'] + eps)

    return df

train_opt9 = add_opt15_features(train_base_fixed)
val_opt9   = add_opt15_features(val_base_fixed)

# ── DEFINE EXACT 24-FEATURE SET ───────────────────────────────────────────────
print("\n[2/3] Defining 24-feature set...")

OPT15_CORE = [
    # Original Landsat bands
    'nir', 'green', 'swir16', 'swir22', 'NDMI', 'MNDWI',
    # Landsat derived
    'NDWI', 'swir_ratio', 'green_nir_ratio',
    # TerraClimate base
    'pet', 'ppt', 'soil', 'tmax', 'tmin',
    # TerraClimate derived
    'temp_range', 'water_balance', 'aridity_index', 'soil_saturation',
    # Temporal
    'month', 'season', 'day_of_year'
]

TC_EXTRA = ['aet', 'def', 'srad']   # proven useful in Opt 2 (0.3509)

FEATS_OPT9 = OPT15_CORE + TC_EXTRA  # 24 features total

# Verify all features exist in both sets
missing_train = [f for f in FEATS_OPT9 if f not in train_opt9.columns]
missing_val   = [f for f in FEATS_OPT9 if f not in val_opt9.columns]
if missing_train: print(f"   ⚠ Missing in train: {missing_train}")
if missing_val:   print(f"   ⚠ Missing in val  : {missing_val}")

FEATS_OPT9 = [f for f in FEATS_OPT9
              if f in train_opt9.columns and f in val_opt9.columns]

print(f"   OPT15 core  : 21 features")
print(f"   TC extras   : {len([f for f in TC_EXTRA if f in FEATS_OPT9])} features (aet, def, srad)")
print(f"   Total       : {len(FEATS_OPT9)} features  (NO synthetic multi-buffer)")

TARGETS = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']

X_train_raw = train_opt9[FEATS_OPT9].copy()
X_val_raw   = val_opt9[FEATS_OPT9].copy()

# Safety NaN fill — should be 0 after KNN fix but guards against edge cases
train_medians = X_train_raw.median()
X_train_raw.fillna(train_medians, inplace=True)
X_val_raw.fillna(train_medians, inplace=True)

nan_count = X_train_raw.isna().sum().sum()
print(f"\n   Residual NaNs: {nan_count}  (should be 0 — KNN fix eliminates spectral NaNs)")

scaler9 = StandardScaler()
X_train9 = scaler9.fit_transform(X_train_raw)
X_val9   = scaler9.transform(X_val_raw)

y_train9      = train_opt9[TARGETS].values
location_ids9 = train_opt9['Location_ID']

print(f"   X_train: {X_train9.shape}  |  X_val: {X_val9.shape}")
print(f"\n✅ Cell 1 complete — ready to train")



OPTIMIZATION 9 — Clean 24-Feature Set on KNN-Fixed Data
Strategy: Opt 8 KNN fix  +  Proven OPT15 features  -  Synthetic buffer noise

[1/3] Engineering proven features on KNN-fixed data...

[2/3] Defining 24-feature set...
   OPT15 core  : 21 features
   TC extras   : 3 features (aet, def, srad)
   Total       : 24 features  (NO synthetic multi-buffer)

   Residual NaNs: 0  (should be 0 — KNN fix eliminates spectral NaNs)
   X_train: (9319, 24)  |  X_val: (200, 24)

✅ Cell 1 complete — ready to train


In [7]:

# ══════════════════════════════════════════════════════════════════════════════
# OPTIMIZATION 9 — CELL 2: Spatial CV + Training + Submission
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 70)
print("OPT 9 CELL 2: Spatial CV + Full Training + Submission")
print("=" * 70)

ET_PARAMS = dict(n_estimators=400, max_depth=20, min_samples_leaf=10,
                 max_features='sqrt', n_jobs=-1)
RF_PARAMS = dict(n_estimators=300, max_depth=18, min_samples_leaf=10,
                 max_features='sqrt', n_jobs=-1)
ET_W, RF_W = 0.65, 0.35
SEEDS      = [42, 100, 200]
gkf        = GroupKFold(n_splits=5)

# ── SPATIAL CV ────────────────────────────────────────────────────────────────
print("\n[1/3] Spatial CV (5-fold GroupKFold by location)...")

cv9 = {}
for i, tname in enumerate(['TA', 'EC', 'DRP']):
    y_t = y_train9[:, i]
    et_folds, rf_folds, blend_folds = [], [], []

    for tr_idx, va_idx in gkf.split(X_train9, y_t, groups=location_ids9):
        et = ExtraTreesRegressor(**ET_PARAMS, random_state=42)
        rf = RandomForestRegressor(**RF_PARAMS, random_state=42)
        et.fit(X_train9[tr_idx], y_t[tr_idx])
        rf.fit(X_train9[tr_idx], y_t[tr_idx])
        p_et = et.predict(X_train9[va_idx])
        p_rf = rf.predict(X_train9[va_idx])
        p_b  = ET_W * p_et + RF_W * p_rf
        et_folds.append(r2_score(y_t[va_idx], p_et))
        rf_folds.append(r2_score(y_t[va_idx], p_rf))
        blend_folds.append(r2_score(y_t[va_idx], p_b))

    cv9[tname] = np.mean(blend_folds)
    print(f"   {tname}: ET={np.mean(et_folds):.4f}  RF={np.mean(rf_folds):.4f}  "
          f"Blend={np.mean(blend_folds):.4f}  "
          f"{[f'{x:.3f}' for x in blend_folds]}")

overall_cv9 = np.mean(list(cv9.values()))

print(f"\n   ┌──────────────────────────────────────────────┐")
print(f"   │  CV Comparison                               │")
print(f"   │  Opt 7  : 0.2217  (best in this notebook)   │")
print(f"   │  Opt 8  : 0.2623  (+0.04 from KNN fix)      │")
print(f"   │  Opt 9  : {overall_cv9:.4f}  ← this run              │")
print(f"   │  Opt 2/5: ~0.31   (leaderboard 0.3499)      │")
print(f"   └──────────────────────────────────────────────┘")

if overall_cv9 >= 0.31:
    print(f"   ✅ Matches or beats Opt 2/5 — strong leaderboard prediction")
elif overall_cv9 >= 0.2623:
    print(f"   ✅ Beats Opt 8 — KNN fix + clean features working")
else:
    print(f"   ⚠ Below Opt 8 — investigate feature pipeline")

# ── FULL 3-SEED ENSEMBLE TRAINING ─────────────────────────────────────────────
print("\n[2/3] Training full 3-seed ensemble on all 9,319 rows...")

val_preds_all = []
for seed in SEEDS:
    seed_preds = np.zeros((len(X_val9), 3))
    for i in range(3):
        y_t = y_train9[:, i]
        et = ExtraTreesRegressor(**ET_PARAMS, random_state=seed)
        rf = RandomForestRegressor(**RF_PARAMS, random_state=seed)
        et.fit(X_train9, y_t)
        rf.fit(X_train9, y_t)
        seed_preds[:, i] = ET_W * et.predict(X_val9) + RF_W * rf.predict(X_val9)
    val_preds_all.append(seed_preds)
    print(f"   Seed {seed} done")

final9 = np.clip(np.mean(val_preds_all, axis=0), 0, None)

# ── PREDICTION HEALTH CHECK ───────────────────────────────────────────────────
print("\n[3/3] Prediction health check + submission...")

benchmarks = {
    'Opt 7  (LB 0.3109)': [103.9, 403.6, 35.0],
    'Opt 15 (LB 0.3469)': [93.0,  282.0, 27.0],
}
print(f"\n   Row 0 comparison (Lat=-32.04, Lon=27.82, Sep-2014):")
print(f"   {'Model':<24} {'TA':>8} {'EC':>8} {'DRP':>8}")
print(f"   {'-'*50}")
for label, vals in benchmarks.items():
    print(f"   {label:<24} {vals[0]:>8.1f} {vals[1]:>8.1f} {vals[2]:>8.1f}")
print(f"   {'Opt 9 (this run)':<24} "
      f"{final9[0,0]:>8.2f} {final9[0,1]:>8.2f} {final9[0,2]:>8.2f}")

print(f"\n   Prediction spread (std should be close to Opt 2/5 levels):")
opt25_std = {'TA': 41.0, 'EC': 158.0, 'DRP': 10.3}
for i, (col, tname) in enumerate(zip(TARGETS, ['TA', 'EC', 'DRP'])):
    p = final9[:, i]
    ref = opt25_std[tname]
    flag = "✅" if p.std() >= ref * 0.7 else "⚠ compressed"
    print(f"   {tname}: min={p.min():.1f}  max={p.max():.1f}  "
          f"mean={p.mean():.2f}  std={p.std():.2f}  "
          f"(Opt2/5 std≈{ref})  {flag}")

# Save submission
submission9 = pd.read_csv('submission_template.csv')
assert len(submission9) == 200
for i, col in enumerate(TARGETS):
    submission9[col] = final9[:, i]
assert submission9[TARGETS].isna().sum().sum() == 0
submission9.to_csv('submission.csv', index=False)

elapsed9 = time.time() - start9
print(f"\n{'=' * 70}")
print(f"OPTIMIZATION 9 COMPLETE")
print(f"{'=' * 70}")
print(f"   Training rows  : {len(X_train9):,}  (clean — KNN fixed 1,085 NaN rows)")
print(f"   Features       : {len(FEATS_OPT9)}  (OPT15 proven set + aet/def/srad)")
print(f"   Dropped        : 45 synthetic multi-buffer features from Opt 8")
print(f"   Overall CV R²  : {overall_cv9:.4f}")
print(f"   Elapsed        : {elapsed9:.0f}s")
print(f"   Saved          : submission_opt9.csv")
print(f"\n   Full submission head:")
print(submission9.head(6).to_string(index=False))

OPT 9 CELL 2: Spatial CV + Full Training + Submission

[1/3] Spatial CV (5-fold GroupKFold by location)...
   TA: ET=0.2910  RF=0.3002  Blend=0.3032  ['0.294', '0.169', '0.314', '0.322', '0.416']
   EC: ET=0.3122  RF=0.3522  Blend=0.3367  ['0.285', '0.390', '0.322', '0.285', '0.402']
   DRP: ET=0.1192  RF=0.1222  Blend=0.1277  ['0.170', '0.126', '0.153', '0.167', '0.023']

   ┌──────────────────────────────────────────────┐
   │  CV Comparison                               │
   │  Opt 7  : 0.2217  (best in this notebook)   │
   │  Opt 8  : 0.2623  (+0.04 from KNN fix)      │
   │  Opt 9  : 0.2559  ← this run              │
   │  Opt 2/5: ~0.31   (leaderboard 0.3499)      │
   └──────────────────────────────────────────────┘
   ⚠ Below Opt 8 — investigate feature pipeline

[2/3] Training full 3-seed ensemble on all 9,319 rows...
   Seed 42 done
   Seed 100 done
   Seed 200 done

[3/3] Prediction health check + submission...

   Row 0 comparison (Lat=-32.04, Lon=27.82, Sep-2014):
   Mode

In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# OPTIMIZATION 10 — CELL 1 (REVISED): Derived Hydrological Proxies
# ══════════════════════════════════════════════════════════════════════════════
#
# q and pdsi not in files — so we derive them from what IS available:
#
#   runoff_proxy  = max(ppt - aet, 0)
#       ppt minus ACTUAL evapotranspiration = water that becomes runoff
#       Better than water_balance (ppt - pet) which uses potential ET not actual
#       DRP washes off in this water — direct physical link
#
#   aet_pet_ratio = aet / (pet + eps)
#       Ranges 0→1. Near 1 = wet (no water stress). Near 0 = drought.
#       This IS a Palmer-type index, just derived from variables we have.
#       Drought (low ratio) → low river flow → concentrated EC + TA
#
# Everything else identical to Opt 9 (0.3489 — personal best):
#   + DRP log1p (new — skew 1.645 → 0.830)
#   + 5 seeds (was 3)
#   + KNN-fixed train_base_fixed / val_base_fixed
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 70)
print("OPTIMIZATION 10 — Derived Hydrological Proxies + DRP log1p")
print("=" * 70)
print("runoff_proxy  = max(ppt - aet, 0)   ← q from available data")
print("aet_pet_ratio = aet / pet            ← pdsi from available data")
print("DRP log1p + 5-seed avg on 0.3489 base")

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
import time

start10 = time.time()

# ── FEATURE ENGINEERING ───────────────────────────────────────────────────────
print("\n[1/4] Engineering features on KNN-fixed data...")

def add_features(df):
    df = df.copy()
    eps = 1e-10
    # Spectral
    if 'nir' in df.columns and 'green' in df.columns:
        df['NDWI']            = (df['green'] - df['nir']) / (df['green'] + df['nir'] + eps)
        df['green_nir_ratio'] = df['green'] / (df['nir'] + eps)
    if 'swir22' in df.columns and 'swir16' in df.columns:
        df['swir_ratio']      = df['swir22'] / (df['swir16'] + eps)
    # TerraClimate derived
    if 'tmax' in df.columns and 'tmin' in df.columns:
        df['temp_range']      = df['tmax'] - df['tmin']
    if 'ppt' in df.columns and 'pet' in df.columns:
        df['water_balance']   = df['ppt'] - df['pet']
        df['aridity_index']   = df['pet'] / (df['ppt'] + eps)
    if 'soil' in df.columns and 'ppt' in df.columns:
        df['soil_saturation'] = df['soil'] / (df['ppt'] + eps)
    # ── NEW: hydrological proxies derived from available variables ──────────
    if 'ppt' in df.columns and 'aet' in df.columns:
        df['runoff_proxy']    = np.maximum(df['ppt'] - df['aet'], 0)
        # max(ppt - actual_ET, 0) — water that has nowhere to go but runoff
        # Better than ppt - pet: actual ET accounts for soil/vegetation limits
    if 'aet' in df.columns and 'pet' in df.columns:
        df['aet_pet_ratio']   = df['aet'] / (df['pet'] + eps)
        # Ranges 0 (extreme drought) → 1 (no water stress, wet conditions)
        # This is the hydrological drought signal — a Palmer proxy
    return df

train_opt10 = add_features(train_base_fixed)
val_opt10   = add_features(val_base_fixed)

# ── DEFINE 26-FEATURE SET ─────────────────────────────────────────────────────
print("\n[2/4] Defining 26-feature set...")

OPT15_CORE = [
    'nir', 'green', 'swir16', 'swir22', 'NDMI', 'MNDWI',
    'NDWI', 'swir_ratio', 'green_nir_ratio',
    'pet', 'ppt', 'soil', 'tmax', 'tmin',
    'temp_range', 'water_balance', 'aridity_index', 'soil_saturation',
    'month', 'season', 'day_of_year'
]
TC_PROVEN  = ['aet', 'def', 'srad']
NEW_HYDRO  = ['runoff_proxy', 'aet_pet_ratio']   # derived from available data

FEATS_10   = OPT15_CORE + TC_PROVEN + NEW_HYDRO  # 26 total

# Availability check
all_ok = True
for f in NEW_HYDRO:
    in_train = f in train_opt10.columns
    in_val   = f in val_opt10.columns
    ok = in_train and in_val
    if not ok: all_ok = False
    print(f"   {f:20s}: {'✅ both' if ok else '❌ MISSING'}")

FEATS_10 = [f for f in FEATS_10
            if f in train_opt10.columns and f in val_opt10.columns]
print(f"   Total features : {len(FEATS_10)}")

# ── PREPARE ARRAYS ────────────────────────────────────────────────────────────
print("\n[3/4] Preparing train/val arrays...")

TARGETS     = ['Total Alkalinity', 'Electrical Conductance',
               'Dissolved Reactive Phosphorus']

X_train_raw = train_opt10[FEATS_10].copy()
X_val_raw   = val_opt10[FEATS_10].copy()

train_medians = X_train_raw.median()
X_train_raw.fillna(train_medians, inplace=True)
X_val_raw.fillna(train_medians, inplace=True)

print(f"   Residual NaNs: {X_train_raw.isna().sum().sum()}")

# Sanity check new features
for f in NEW_HYDRO:
    if f in FEATS_10:
        col = X_train_raw[f]
        print(f"   {f}: min={col.min():.3f}  mean={col.mean():.3f}  "
              f"max={col.max():.3f}  zeros={(col==0).sum()}")

scaler10  = StandardScaler()
X_train10 = scaler10.fit_transform(X_train_raw)
X_val10   = scaler10.transform(X_val_raw)

y_raw = train_opt10[TARGETS].values

# ── PER-TARGET TARGET ARRAYS ──────────────────────────────────────────────────
print("\n[4/4] Per-target arrays (DRP gets log1p)...")

y_TA  = y_raw[:, 0]
y_EC  = y_raw[:, 1]
y_DRP_raw = y_raw[:, 2]
y_DRP     = np.log1p(y_DRP_raw)

print(f"   DRP skew  before: {pd.Series(y_DRP_raw).skew():.3f}")
print(f"   DRP skew  after : {pd.Series(y_DRP).skew():.3f}")

location_ids10 = train_opt10['Location_ID']

print(f"\n   X_train: {X_train10.shape}  |  X_val: {X_val10.shape}")
print(f"\n✅ Cell 1 complete — {len(FEATS_10)} features ready")

OPTIMIZATION 10 — Derived Hydrological Proxies + DRP log1p
runoff_proxy  = max(ppt - aet, 0)   ← q from available data
aet_pet_ratio = aet / pet            ← pdsi from available data
DRP log1p + 5-seed avg on 0.3489 base

[1/4] Engineering features on KNN-fixed data...

[2/4] Defining 26-feature set...
   runoff_proxy        : ✅ both
   aet_pet_ratio       : ✅ both
   Total features : 26

[3/4] Preparing train/val arrays...
   Residual NaNs: 0
   runoff_proxy: min=0.000  mean=6.444  max=242.000  zeros=1605
   aet_pet_ratio: min=0.000  mean=0.408  max=1.000  zeros=9

[4/4] Per-target arrays (DRP gets log1p)...
   DRP skew  before: 1.645
   DRP skew  after : 0.830

   X_train: (9319, 26)  |  X_val: (200, 26)

✅ Cell 1 complete — 26 features ready


In [14]:

# ══════════════════════════════════════════════════════════════════════════════
# OPTIMIZATION 10 — CELL 2: Per-Target CV + 5-Seed Ensemble + Submission
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 70)
print("OPT 10 CELL 2: Per-Target Spatial CV + 5-Seed Ensemble")
print("=" * 70)

ET_PARAMS = dict(n_estimators=400, max_depth=20, min_samples_leaf=10,
                 max_features='sqrt', n_jobs=-1)
RF_PARAMS = dict(n_estimators=300, max_depth=18, min_samples_leaf=10,
                 max_features='sqrt', n_jobs=-1)
ET_W, RF_W = 0.65, 0.35
SEEDS      = [42, 100, 200, 300, 400]    # 5 seeds — was 3 in Opt 9
gkf        = GroupKFold(n_splits=5)

# Per-target config: (name, y_array, log_transform)
TARGET_CFG = [
    ('TA',  y_TA,  False),   # standard — normally distributed
    ('EC',  y_EC,  False),   # standard — normally distributed
    ('DRP', y_DRP, True),    # log1p — right-skewed, treat separately
]

# ── SPATIAL CV — PER TARGET ───────────────────────────────────────────────────
print(f"\n[1/3] Spatial CV (5-fold GroupKFold, per-target)...")
print(f"      Architecture: {ET_W*100:.0f}% ET + {RF_W*100:.0f}% RF | "
      f"leaf=10 | {len(SEEDS)} seeds")

cv10 = {}
for tname, y_t, log_t in TARGET_CFG:
    et_folds, rf_folds, blend_folds = [], [], []

    for tr_idx, va_idx in gkf.split(X_train10, y_t, groups=location_ids10):
        et = ExtraTreesRegressor(**ET_PARAMS, random_state=42)
        rf = RandomForestRegressor(**RF_PARAMS, random_state=42)
        et.fit(X_train10[tr_idx], y_t[tr_idx])
        rf.fit(X_train10[tr_idx], y_t[tr_idx])
        p_et = et.predict(X_train10[va_idx])
        p_rf = rf.predict(X_train10[va_idx])
        p_b  = ET_W * p_et + RF_W * p_rf

        # For DRP: score in original space (expm1 back)
        if log_t:
            y_va_orig = np.expm1(y_t[va_idx])
            p_b_orig  = np.expm1(np.clip(p_b, 0, None))
            p_et_orig = np.expm1(np.clip(p_et, 0, None))
            p_rf_orig = np.expm1(np.clip(p_rf, 0, None))
            et_folds.append(r2_score(y_va_orig, p_et_orig))
            rf_folds.append(r2_score(y_va_orig, p_rf_orig))
            blend_folds.append(r2_score(y_va_orig, p_b_orig))
        else:
            et_folds.append(r2_score(y_t[va_idx], p_et))
            rf_folds.append(r2_score(y_t[va_idx], p_rf))
            blend_folds.append(r2_score(y_t[va_idx], p_b))

    cv10[tname] = np.mean(blend_folds)
    transform_tag = " [log1p]" if log_t else ""
    print(f"   {tname}{transform_tag}: "
          f"ET={np.mean(et_folds):.4f}  RF={np.mean(rf_folds):.4f}  "
          f"Blend={np.mean(blend_folds):.4f}  "
          f"{[f'{x:.3f}' for x in blend_folds]}")

overall_cv10 = np.mean(list(cv10.values()))

print(f"\n   ┌─────────────────────────────────────────────────────┐")
print(f"   │  CV Progression                                     │")
print(f"   │  Opt 15 (LB 0.3469):  CV ~0.31                     │")
print(f"   │  Opt 9  (LB 0.3489):  CV  0.2559                   │")
print(f"   │  Opt 10 (this run) :  CV  {overall_cv10:.4f}                   │")
print(f"   └─────────────────────────────────────────────────────┘")

# DRP-specific diagnostics
print(f"\n   DRP CV breakdown (scored in original space, not log):")
print(f"Fold R²: {[f'{x:.3f}' for x in blend_folds]}")
print(f"   Mean: {cv10['DRP']:.4f}  (Opt 9 had 0.1277 — log1p should lift this)")

# ── 5-SEED FULL TRAINING ──────────────────────────────────────────────────────
print(f"\n[2/3] Training 5-seed ensemble on all 9,319 rows...")
print(f"      Per-target: TA/EC standard | DRP log1p → expm1")

val_preds_all = []
for seed in SEEDS:
    seed_preds = np.zeros((len(X_val10), 3))
    for i, (tname, y_t, log_t) in enumerate(TARGET_CFG):
        et = ExtraTreesRegressor(**ET_PARAMS, random_state=seed)
        rf = RandomForestRegressor(**RF_PARAMS, random_state=seed)
        et.fit(X_train10, y_t)
        rf.fit(X_train10, y_t)
        p = ET_W * et.predict(X_val10) + RF_W * rf.predict(X_val10)
        # Back-transform DRP
        seed_preds[:, i] = np.expm1(np.clip(p, 0, None)) if log_t else p
    val_preds_all.append(seed_preds)
    print(f"   Seed {seed} done")

final10 = np.clip(np.mean(val_preds_all, axis=0), 0, None)

# ── PREDICTION HEALTH CHECK ───────────────────────────────────────────────────
print(f"\n[3/3] Prediction health check...")

print(f"\n   Row 0 comparison (Lat=-32.04, Lon=27.82, Sep-2014):")
benchmarks = {
    'Opt 9  (LB 0.3489)': [96.16,  292.16, 27.78],
    'Opt 15 (LB 0.3469)': [93.0,   282.0,  27.0],
}
print(f"   {'Model':<24} {'TA':>8} {'EC':>8} {'DRP':>8}")
print(f"   {'-'*52}")
for label, vals in benchmarks.items():
    print(f"   {label:<24} {vals[0]:>8.1f} {vals[1]:>8.1f} {vals[2]:>8.1f}")
print(f"   {'Opt 10 (this run)':<24} "
      f"{final10[0,0]:>8.2f} {final10[0,1]:>8.2f} {final10[0,2]:>8.2f}")

print(f"\n   Spread check (goal: match Opt 9 levels or better):")
opt9_std = {'TA': 39.36, 'EC': 145.97, 'DRP': 10.67}
for i, tname in enumerate(['TA', 'EC', 'DRP']):
    p    = final10[:, i]
    ref  = opt9_std[tname]
    flag = "✅" if p.std() >= ref * 0.80 else "⚠ compressed"
    print(f"   {tname}: min={p.min():.1f}  max={p.max():.1f}  "
          f"mean={p.mean():.2f}  std={p.std():.2f}  "
          f"(Opt9 std={ref})  {flag}")

# ── FEATURE IMPORTANCE — q and pdsi check ────────────────────────────────────
print(f"\n   Feature importance — validating q and pdsi contribute:")
et_final = ExtraTreesRegressor(**ET_PARAMS, random_state=42)
et_final.fit(X_train10, y_TA)   # TA importance as proxy
imp = pd.Series(et_final.feature_importances_,
                index=FEATS_10).sort_values(ascending=False)
print(f"   Top 10 features:")
for feat, val in imp.head(10).items():
    marker = " ← NEW" if feat in ['q', 'pdsi'] else ""
    print(f"   {feat:22s}: {val:.4f}{marker}")

# q and pdsi rank
for f in ['q', 'pdsi']:
    if f in imp.index:
        rank = list(imp.index).index(f) + 1
        print(f"   {f} ranks #{rank} of {len(FEATS_10)} features  "
              f"(importance={imp[f]:.4f})")

# ── SAVE SUBMISSION ───────────────────────────────────────────────────────────
submission10 = pd.read_csv('submission_template.csv')
assert len(submission10) == 200
for i, col in enumerate(TARGETS):
    submission10[col] = final10[:, i]
assert submission10[TARGETS].isna().sum().sum() == 0
submission10.to_csv('submission.csv', index=False)

elapsed10 = time.time() - start10
print(f"\n{'=' * 70}")
print(f"OPTIMIZATION 10 COMPLETE")
print(f"{'=' * 70}")
print(f"   Training rows  : {len(X_train10):,}")
print(f"   Features       : {len(FEATS_10)}  (OPT15 + aet/def/srad + q/pdsi)")
print(f"   New features   : q (runoff), pdsi (drought) — hydrological state")
print(f"   DRP transform  : log1p → train, expm1 → predict (first isolated use)")
print(f"   Seeds          : {len(SEEDS)} (was 3 in Opt 9)")
print(f"   Overall CV R²  : {overall_cv10:.4f}")
print(f"   Elapsed        : {elapsed10:.0f}s")
print(f"   Saved          : submission_opt10.csv")
print(f"\n   Full submission head:")
print(submission10.head(6).to_string(index=False))
print(f"\n   Submit submission_opt10.csv — this is the final shot.")

OPT 10 CELL 2: Per-Target Spatial CV + 5-Seed Ensemble

[1/3] Spatial CV (5-fold GroupKFold, per-target)...
      Architecture: 65% ET + 35% RF | leaf=10 | 5 seeds
   TA: ET=0.3150  RF=0.3061  Blend=0.3190  ['0.297', '0.173', '0.352', '0.340', '0.434']
   EC: ET=0.3419  RF=0.3621  Blend=0.3572  ['0.311', '0.426', '0.333', '0.292', '0.423']
   DRP [log1p]: ET=0.0021  RF=0.0253  Blend=0.0125  ['0.061', '0.074', '-0.013', '-0.021', '-0.039']

   ┌─────────────────────────────────────────────────────┐
   │  CV Progression                                     │
   │  Opt 15 (LB 0.3469):  CV ~0.31                     │
   │  Opt 9  (LB 0.3489):  CV  0.2559                   │
   │  Opt 10 (this run) :  CV  0.2296                   │
   └─────────────────────────────────────────────────────┘

   DRP CV breakdown (scored in original space, not log):
Fold R²: ['0.061', '0.074', '-0.013', '-0.021', '-0.039']
   Mean: 0.0125  (Opt 9 had 0.1277 — log1p should lift this)

[2/3] Training 5-seed ensem

In [16]:
# ══════════════════════════════════════════════════════════════════════════════
# OPTIMIZATION 11 — Consolidation of Best Practices
# ══════════════════════════════════════════════════════════════════════════════
#
# ROOT CAUSE ANALYSIS OF OPT 10 COLLAPSE (0.3489 → 0.253):
#
#   Change 1 — runoff_proxy = max(ppt - aet, 0)
#     PROBLEM: ppt and aet are ALREADY individual features. This is a
#     deterministic function of two existing columns → pure multicollinearity.
#     Trees can already learn "if ppt > aet: X" from raw features directly.
#     Adding this shifts feature importances without adding any information.
#
#   Change 2 — aet_pet_ratio = aet / pet
#     PROBLEM: Same issue. aet and pet are both already in the feature set.
#     The ratio contains zero information beyond what the raw values provide.
#     Another multicollinear derived feature that adds noise to splits.
#
#   Change 3 — DRP log1p target transform
#     PROBLEM: Trees split on quantiles, not values — they are scale-invariant.
#     log1p DOES compress the target but does NOT improve tree splits.
#     What it DOES do: expm1(predicted_log) amplifies errors on high-DRP samples.
#     High-DRP samples tend to be at unseen spatial locations (the hard cases).
#     Amplifying error exactly where spatial generalization is weakest = disaster.
#
#   Change 4 — 5 seeds instead of 3
#     This is FINE. Cannot cause a 0.096 drop. Worth keeping permanently.
#
# OPT 11 STRATEGY:
#   Revert the 3 bad changes. Keep the 1 good change (5 seeds).
#   Additionally, test temporal lags of ppt/soil/aet at lag-1 and lag-2 months
#   — these are GENUINELY new causal information (rainfall 4-8 weeks ago drives
#   today's DRP and EC), computed within each location via groupby.shift().
#   Lags are only kept if they beat the Opt 9 baseline in spatial CV.
#
# Carries over from Opt 8 Cell 1 (must be in memory):
#   train_base_fixed — 9,319 rows, KNN-fixed orig Landsat NaNs
#   val_base_fixed   — 200 rows,   KNN-fixed orig Landsat NaNs
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 70)
print("OPTIMIZATION 11 — Safe Consolidation + Temporal Lag Validation")
print("=" * 70)
print()
print("Revert: DRP log1p ❌  runoff_proxy ❌  aet_pet_ratio ❌")
print("Keep  : KNN fix ✅  24-feature set ✅  5 seeds ✅")
print("Test  : ppt/soil/aet lag-1 and lag-2 (genuine causal hydrology signal)")
print()

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
import warnings
import time

warnings.filterwarnings('ignore')
start11 = time.time()

# ══════════════════════════════════════════════════════════════════════════════
# CELL 1 — FEATURE ENGINEERING + CV COMPARISON
# ══════════════════════════════════════════════════════════════════════════════

# ── BASE FEATURE ENGINEERING (identical to Opt 9) ────────────────────────────
print("[1/5] Engineering Opt 9 base features on KNN-fixed data...")

def add_base_features(df):
    """Exact Opt 9 proven feature set — no additions, no regressions."""
    df = df.copy()
    eps = 1e-10
    if 'nir' in df.columns and 'green' in df.columns:
        df['NDWI']            = (df['green'] - df['nir']) / (df['green'] + df['nir'] + eps)
        df['green_nir_ratio'] = df['green'] / (df['nir'] + eps)
    if 'swir22' in df.columns and 'swir16' in df.columns:
        df['swir_ratio']      = df['swir22'] / (df['swir16'] + eps)
    if 'tmax' in df.columns and 'tmin' in df.columns:
        df['temp_range']      = df['tmax'] - df['tmin']
    if 'ppt' in df.columns and 'pet' in df.columns:
        df['water_balance']   = df['ppt'] - df['pet']
        df['aridity_index']   = df['pet'] / (df['ppt'] + eps)
    if 'soil' in df.columns and 'ppt' in df.columns:
        df['soil_saturation'] = df['soil'] / (df['ppt'] + eps)
    return df

train11 = add_base_features(train_base_fixed)
val11   = add_base_features(val_base_fixed)

# ── TEMPORAL LAG ENGINEERING ──────────────────────────────────────────────────
print("[2/5] Engineering temporal lags (ppt, soil, aet at lag-1, lag-2)...")
print()
print("  Rationale: Water quality lags climate forcing by 1-2 months.")
print("  DRP (phosphorus) is driven by surface runoff from rain 4-8 weeks ago.")
print("  EC (conductivity) is driven by low-flow periods after dry months.")
print("  These add GENUINELY new information — not linear combos of existing features.")
print("  Lags are computed within each location (groupby Location_ID) to avoid")
print("  cross-site contamination. Val NaN boundary rows → filled with train median.")
print()

LAG_VARS  = ['ppt', 'soil', 'aet']
LAG_STEPS = [1, 2]

def ensure_location_id(df):
    """Add Location_ID if missing — same construction as train."""
    df = df.copy()
    if 'Location_ID' not in df.columns:
        df['Location_ID'] = (df['Latitude'].round(4).astype(str)
                             + '_' + df['Longitude'].round(4).astype(str))
    return df

def add_temporal_lags(df, lag_vars, lag_steps, sort_col='Sample Date'):
    df = ensure_location_id(df).copy()
    # sort_col may be missing on val template — guard against it
    sort_cols = ['Location_ID', sort_col] if sort_col in df.columns else ['Location_ID']
    df_sorted = df.sort_values(sort_cols).copy()
    for var in lag_vars:
        if var not in df_sorted.columns:
            print(f"  ⚠ {var} not found — skipping")
            continue
        for lag in lag_steps:
            df_sorted[f'{var}_lag{lag}'] = (
                df_sorted.groupby('Location_ID')[var].shift(lag)
            )
    df_result = df_sorted.reindex(df.index)
    lag_cols = [f'{v}_lag{l}' for v in lag_vars
                if v in df.columns for l in lag_steps]
    lag_cols = [c for c in lag_cols if c in df_result.columns]
    print("  Lag NaN counts (boundary rows — filled with median later):")
    for c in lag_cols:
        n = df_result[c].isna().sum()
        print(f"    {c}: {n} ({100*n/len(df_result):.1f}%)")
    return df_result

train11_lag = add_temporal_lags(train11, LAG_VARS, LAG_STEPS)
# Val has 1 row per location — lags will all be NaN, filled with train medians.
# This is correct: no historical val climate is available, so the model uses
# the training population mean for lag features on val rows.
val11_lag   = add_temporal_lags(val11,   LAG_VARS, LAG_STEPS)

lag_feature_names = [f'{v}_lag{l}' for v in LAG_VARS
                     for l in LAG_STEPS
                     if f'{v}_lag{l}' in train11_lag.columns]
print(f"\n  Generated {len(lag_feature_names)} lag features: {lag_feature_names}")

# ── DEFINE FEATURE SETS FOR CV COMPARISON ────────────────────────────────────
print("\n[3/5] Defining feature sets for CV comparison...")

OPT15_CORE = [
    'nir', 'green', 'swir16', 'swir22', 'NDMI', 'MNDWI',
    'NDWI', 'swir_ratio', 'green_nir_ratio',
    'pet', 'ppt', 'soil', 'tmax', 'tmin',
    'temp_range', 'water_balance', 'aridity_index', 'soil_saturation',
    'month', 'season', 'day_of_year'
]
TC_PROVEN = ['aet', 'def', 'srad']
FEATS_A   = OPT15_CORE + TC_PROVEN            # 24 — exact Opt 9
FEATS_B   = FEATS_A + lag_feature_names       # 24 + 6 = 30

# Filter to available columns in both train and val
FEATS_A = [f for f in FEATS_A if f in train11_lag.columns and f in val11_lag.columns]
FEATS_B = [f for f in FEATS_B if f in train11_lag.columns and f in val11_lag.columns]

print(f"  Set A (Opt 9 baseline) : {len(FEATS_A)} features")
print(f"  Set B (+ all lags)     : {len(FEATS_B)} features")

# ── PREPARE ARRAYS FOR BOTH FEATURE SETS ─────────────────────────────────────
print("\n[4/5] Preparing scaled arrays for both feature sets...")

TARGETS = ['Total Alkalinity', 'Electrical Conductance',
           'Dissolved Reactive Phosphorus']

def prepare_arrays(train_df, val_df, feature_list):
    X_tr = train_df[feature_list].copy()
    X_va = val_df[feature_list].copy()
    medians = X_tr.median()
    X_tr.fillna(medians, inplace=True)
    X_va.fillna(medians, inplace=True)
    assert X_tr.isna().sum().sum() == 0, "NaNs remain in train"
    assert X_va.isna().sum().sum() == 0, "NaNs remain in val"
    sc = StandardScaler()
    return sc.fit_transform(X_tr), sc.transform(X_va), sc

X_trA, X_vaA, scA = prepare_arrays(train11_lag, val11_lag, FEATS_A)
X_trB, X_vaB, scB = prepare_arrays(train11_lag, val11_lag, FEATS_B)

y_train11      = train11_lag[TARGETS].values
location_ids11 = train11_lag['Location_ID']

print(f"  Set A: X_train={X_trA.shape}  X_val={X_vaA.shape}")
print(f"  Set B: X_train={X_trB.shape}  X_val={X_vaB.shape}")

# ── CV COMPARISON: SET A vs SET B ────────────────────────────────────────────
print("\n[5/5] Spatial CV comparison (5-fold GroupKFold)...")
print("  Architecture: ET 65% + RF 35% | leaf=10 | seed=42")
print()

ET_PARAMS = dict(n_estimators=400, max_depth=20, min_samples_leaf=10,
                 max_features='sqrt', n_jobs=-1)
RF_PARAMS = dict(n_estimators=300, max_depth=18, min_samples_leaf=10,
                 max_features='sqrt', n_jobs=-1)
ET_W, RF_W = 0.65, 0.35
gkf = GroupKFold(n_splits=5)

def run_cv(X_tr, y_arr, groups, label):
    cv_per_target = {}
    for i, tname in enumerate(['TA', 'EC', 'DRP']):
        y_t = y_arr[:, i]
        folds = []
        for tr_idx, va_idx in gkf.split(X_tr, y_t, groups=groups):
            et = ExtraTreesRegressor(**ET_PARAMS, random_state=42)
            rf = RandomForestRegressor(**RF_PARAMS, random_state=42)
            et.fit(X_tr[tr_idx], y_t[tr_idx])
            rf.fit(X_tr[tr_idx], y_t[tr_idx])
            p = ET_W * et.predict(X_tr[va_idx]) + RF_W * rf.predict(X_tr[va_idx])
            folds.append(r2_score(y_t[va_idx], p))
        cv_per_target[tname] = np.mean(folds)
        print(f"  [{label}] {tname}: {np.mean(folds):.4f}  "
              f"folds={[f'{x:.3f}' for x in folds]}")
    overall = np.mean(list(cv_per_target.values()))
    print(f"  [{label}] OVERALL: {overall:.4f}")
    print()
    return overall, cv_per_target

print("--- Set A: Opt 9 baseline (24 features) ---")
cv_A, cv_A_targets = run_cv(X_trA, y_train11, location_ids11, 'A')

print("--- Set B: + temporal lags (30 features) ---")
cv_B, cv_B_targets = run_cv(X_trB, y_train11, location_ids11, 'B')

# ── DECISION LOGIC ────────────────────────────────────────────────────────────
print("=" * 70)
print("CV COMPARISON RESULT")
print("=" * 70)
print(f"  Set A (Opt 9 baseline) : {cv_A:.4f}")
print(f"  Set B (+ lags)         : {cv_B:.4f}")
print(f"  Delta B-A              : {cv_B - cv_A:+.4f}")
print()

for t in ['TA', 'EC', 'DRP']:
    delta  = cv_B_targets[t] - cv_A_targets[t]
    winner = 'B (lags)' if delta > 0 else 'A (no lags)'
    print(f"  {t}: A={cv_A_targets[t]:.4f}  B={cv_B_targets[t]:.4f}  "
          f"delta={delta:+.4f}  → {winner}")
print()

# Lags must clear a +0.002 threshold to be worth the added complexity risk
THRESHOLD = 0.002

if cv_B >= cv_A + THRESHOLD:
    print(f"  ✅ DECISION: USE SET B — lags improve CV by ≥{THRESHOLD}")
    FEATS_FINAL = FEATS_B
    X_tr_FINAL  = X_trB
    X_va_FINAL  = X_vaB
    SET_LABEL   = f'B (24 + {len(lag_feature_names)} lags = {len(FEATS_B)} features)'
else:
    if cv_B > cv_A:
        print(f"  ⚠ DECISION: USE SET A — lag improvement {cv_B-cv_A:+.4f} < threshold {THRESHOLD}")
    else:
        print(f"  ✅ DECISION: USE SET A — lags hurt or neutral in CV")
    FEATS_FINAL = FEATS_A
    X_tr_FINAL  = X_trA
    X_va_FINAL  = X_vaA
    SET_LABEL   = 'A (24 features — Opt 9 baseline)'

print(f"\n  Final feature set: {SET_LABEL}")
print(f"  Proceeding to 5-seed full training...")


OPTIMIZATION 11 — Safe Consolidation + Temporal Lag Validation

Revert: DRP log1p ❌  runoff_proxy ❌  aet_pet_ratio ❌
Keep  : KNN fix ✅  24-feature set ✅  5 seeds ✅
Test  : ppt/soil/aet lag-1 and lag-2 (genuine causal hydrology signal)

[1/5] Engineering Opt 9 base features on KNN-fixed data...
[2/5] Engineering temporal lags (ppt, soil, aet at lag-1, lag-2)...

  Rationale: Water quality lags climate forcing by 1-2 months.
  DRP (phosphorus) is driven by surface runoff from rain 4-8 weeks ago.
  EC (conductivity) is driven by low-flow periods after dry months.
  These add GENUINELY new information — not linear combos of existing features.
  Lags are computed within each location (groupby Location_ID) to avoid
  cross-site contamination. Val NaN boundary rows → filled with train median.

  Lag NaN counts (boundary rows — filled with median later):
    ppt_lag1: 162 (1.7%)
    ppt_lag2: 324 (3.5%)
    soil_lag1: 162 (1.7%)
    soil_lag2: 324 (3.5%)
    aet_lag1: 162 (1.7%)
    aet_lag2: 

In [17]:

# ══════════════════════════════════════════════════════════════════════════════
# CELL 2 — 5-SEED ENSEMBLE TRAINING + SUBMISSION
# ══════════════════════════════════════════════════════════════════════════════

print()
print("=" * 70)
print("OPT 11 CELL 2: 5-Seed Ensemble + Submission")
print("=" * 70)
print(f"  Feature set  : {SET_LABEL}")
print(f"  Features     : {len(FEATS_FINAL)}")
print(f"  Targets      : Raw TA, EC, DRP — NO log transform")
print(f"  Seeds        : 5  (was 3 in Opt 9)")
print(f"  Architecture : ET 65% + RF 35% | leaf=10 | depth 20/18")
print()

SEEDS = [42, 100, 200, 300, 400]

# ── 5-SEED FULL TRAINING ──────────────────────────────────────────────────────
print("[1/3] Training 5-seed ensemble on all 9,319 rows (raw targets)...")
print()

val_preds_all = []
for seed in SEEDS:
    seed_preds = np.zeros((len(X_va_FINAL), 3))
    for i, tname in enumerate(['TA', 'EC', 'DRP']):
        y_t = y_train11[:, i]   # raw — no log transform
        et  = ExtraTreesRegressor(**ET_PARAMS, random_state=seed)
        rf  = RandomForestRegressor(**RF_PARAMS, random_state=seed)
        et.fit(X_tr_FINAL, y_t)
        rf.fit(X_tr_FINAL, y_t)
        seed_preds[:, i] = ET_W * et.predict(X_va_FINAL) + RF_W * rf.predict(X_va_FINAL)
    val_preds_all.append(seed_preds)
    print(f"  Seed {seed} done")

# Average across seeds; clip to 0 (negative water quality is physically impossible)
final11 = np.clip(np.mean(val_preds_all, axis=0), 0, None)

# ── PREDICTION HEALTH CHECK ───────────────────────────────────────────────────
print()
print("[2/3] Prediction health check...")

benchmarks = {
    'Opt 9  (LB 0.3489)': [96.16,  292.16, 27.78],
    'Opt 15 (LB 0.3469)': [93.0,   282.0,  27.0 ],
}
print(f"\n  Row 0 comparison (Lat=-32.04, Lon=27.82, Sep-2014):")
print(f"  {'Model':<26} {'TA':>8} {'EC':>8} {'DRP':>8}")
print(f"  {'-'*54}")
for label, vals in benchmarks.items():
    print(f"  {label:<26} {vals[0]:>8.1f} {vals[1]:>8.1f} {vals[2]:>8.1f}")
print(f"  {'Opt 11 (this run)':<26} "
      f"{final11[0,0]:>8.2f} {final11[0,1]:>8.2f} {final11[0,2]:>8.2f}")

print()
print("  Spread check (goal: match or exceed Opt 9 levels):")
opt9_std = {'TA': 39.36, 'EC': 145.97, 'DRP': 10.67}
for i, tname in enumerate(['TA', 'EC', 'DRP']):
    p    = final11[:, i]
    ref  = opt9_std[tname]
    flag = "✅" if p.std() >= ref * 0.80 else "⚠ COMPRESSED"
    print(f"  {tname}: min={p.min():.1f}  max={p.max():.1f}  "
          f"mean={p.mean():.2f}  std={p.std():.2f}  "
          f"(Opt9 std={ref})  {flag}")

if (final11 < 0).sum() == 0:
    print("\n  ✅ No negative predictions")
else:
    print(f"\n  ⚠ Unexpected negatives remaining: {(final11 < 0).sum()}")

# ── SAVE SUBMISSION ───────────────────────────────────────────────────────────
print()
print("[3/3] Saving submission...")

submission11 = pd.read_csv('submission_template.csv')
assert len(submission11) == 200
for i, col in enumerate(TARGETS):
    submission11[col] = final11[:, i]
assert submission11[TARGETS].isna().sum().sum() == 0
submission11.to_csv('submission.csv', index=False)

elapsed11 = time.time() - start11
print()
print("=" * 70)
print("OPTIMIZATION 11 COMPLETE")
print("=" * 70)
print(f"  Feature set    : {SET_LABEL}")
print(f"  Features       : {len(FEATS_FINAL)}")
print(f"  Training rows  : {len(X_tr_FINAL):,}")
print(f"  Seeds          : {len(SEEDS)}  (was 3 in Opt 9)")
print(f"  DRP transform  : NONE — raw targets throughout")
print(f"  Hydro proxies  : REMOVED — runoff_proxy & aet_pet_ratio were multicollinear")
print(f"  CV Set A       : {cv_A:.4f}  (Opt 9 baseline)")
print(f"  CV Set B       : {cv_B:.4f}  (+ temporal lags)")
print(f"  Selected       : {'B' if FEATS_FINAL == FEATS_B else 'A'}  — "
      f"{'lags helped' if FEATS_FINAL == FEATS_B else 'lags did not pass threshold'}")
print(f"  LB reference   : Opt 9 = 0.3489 | Opt 15 = 0.3469")
print(f"  Elapsed        : {elapsed11:.0f}s")
print(f"  Saved          : submission.csv")
print()
print(submission11.head(6).to_string(index=False))


OPT 11 CELL 2: 5-Seed Ensemble + Submission
  Feature set  : B (24 + 6 lags = 30 features)
  Features     : 30
  Targets      : Raw TA, EC, DRP — NO log transform
  Seeds        : 5  (was 3 in Opt 9)
  Architecture : ET 65% + RF 35% | leaf=10 | depth 20/18

[1/3] Training 5-seed ensemble on all 9,319 rows (raw targets)...

  Seed 42 done
  Seed 100 done
  Seed 200 done
  Seed 300 done
  Seed 400 done

[2/3] Prediction health check...

  Row 0 comparison (Lat=-32.04, Lon=27.82, Sep-2014):
  Model                            TA       EC      DRP
  ------------------------------------------------------
  Opt 9  (LB 0.3489)             96.2    292.2     27.8
  Opt 15 (LB 0.3469)             93.0    282.0     27.0
  Opt 11 (this run)             94.31   298.83    28.28

  Spread check (goal: match or exceed Opt 9 levels):
  TA: min=32.5  max=200.1  mean=90.74  std=41.91  (Opt9 std=39.36)  ✅
  EC: min=182.3  max=814.6  mean=406.49  std=161.13  (Opt9 std=145.97)  ✅
  DRP: min=15.3  max=54.1  

In [18]:
# ══════════════════════════════════════════════════════════════════════════════
# OPTIMIZATION 12 — Extended Temporal Lags (lag-3) + pet lag
# ══════════════════════════════════════════════════════════════════════════════
#
# Opt 11 result: LB 0.3679 (was 0.3489) — lags added +0.019
# CV gap held perfectly: predicted 0.3675, got 0.3679.
#
# What to try next:
#   1. lag-3 for ppt/soil/aet — the CV gain was clean across all folds,
#      no sign of diminishing returns yet. 3-month memory is physically
#      reasonable for baseflow and soil moisture persistence.
#   2. Add pet to the lag variables — potential ET drives water stress
#      with a similar temporal memory to soil moisture.
#   3. Same CV gate: only keep what beats Opt 11 baseline (0.2775) by ≥0.002
#
# Carries over from Opt 8 Cell 1 (must be in memory):
#   train_base_fixed, val_base_fixed
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 70)
print("OPTIMIZATION 12 — Extended Lags: lag-3 + pet lags")
print("=" * 70)
print()
print("Base  : Opt 11 (LB 0.3679) — 30 features, 5 seeds, lags 1-2")
print("Test A: Opt 11 baseline (lag 1-2, ppt/soil/aet)  — 30 features")
print("Test B: + lag-3 for ppt/soil/aet                 — 33 features")
print("Test C: + lag 1-3 for pet as well                — 36 features")
print("CV gate: must beat 0.2775 by ≥0.002 to be kept")
print()

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
import warnings
import time

warnings.filterwarnings('ignore')
start12 = time.time()

TARGETS = ['Total Alkalinity', 'Electrical Conductance',
           'Dissolved Reactive Phosphorus']
ET_PARAMS = dict(n_estimators=400, max_depth=20, min_samples_leaf=10,
                 max_features='sqrt', n_jobs=-1)
RF_PARAMS = dict(n_estimators=300, max_depth=18, min_samples_leaf=10,
                 max_features='sqrt', n_jobs=-1)
ET_W, RF_W = 0.65, 0.35
gkf = GroupKFold(n_splits=5)

# ── BASE FEATURE ENGINEERING ──────────────────────────────────────────────────
print("[1/5] Engineering base features...")

def add_base_features(df):
    df = df.copy()
    eps = 1e-10
    if 'nir' in df.columns and 'green' in df.columns:
        df['NDWI']            = (df['green'] - df['nir']) / (df['green'] + df['nir'] + eps)
        df['green_nir_ratio'] = df['green'] / (df['nir'] + eps)
    if 'swir22' in df.columns and 'swir16' in df.columns:
        df['swir_ratio']      = df['swir22'] / (df['swir16'] + eps)
    if 'tmax' in df.columns and 'tmin' in df.columns:
        df['temp_range']      = df['tmax'] - df['tmin']
    if 'ppt' in df.columns and 'pet' in df.columns:
        df['water_balance']   = df['ppt'] - df['pet']
        df['aridity_index']   = df['pet'] / (df['ppt'] + eps)
    if 'soil' in df.columns and 'ppt' in df.columns:
        df['soil_saturation'] = df['soil'] / (df['ppt'] + eps)
    return df

train12 = add_base_features(train_base_fixed)
val12   = add_base_features(val_base_fixed)

# ── TEMPORAL LAG ENGINEERING ──────────────────────────────────────────────────
print("[2/5] Engineering temporal lags...")

def ensure_location_id(df):
    df = df.copy()
    if 'Location_ID' not in df.columns:
        df['Location_ID'] = (df['Latitude'].round(4).astype(str)
                             + '_' + df['Longitude'].round(4).astype(str))
    return df

def add_temporal_lags(df, lag_vars, lag_steps, sort_col='Sample Date'):
    df = ensure_location_id(df).copy()
    sort_cols = ['Location_ID', sort_col] if sort_col in df.columns else ['Location_ID']
    df_sorted = df.sort_values(sort_cols).copy()
    for var in lag_vars:
        if var not in df_sorted.columns:
            continue
        for lag in lag_steps:
            df_sorted[f'{var}_lag{lag}'] = (
                df_sorted.groupby('Location_ID')[var].shift(lag)
            )
    return df_sorted.reindex(df.index)

# All three candidate lag sets computed once
LAG_VARS_ABC = ['ppt', 'soil', 'aet', 'pet']
LAG_STEPS_C  = [1, 2, 3]

train12_lag = add_temporal_lags(train12, LAG_VARS_ABC, LAG_STEPS_C)
val12_lag   = add_temporal_lags(val12,   LAG_VARS_ABC, LAG_STEPS_C)
print("  Done")

# ── DEFINE FEATURE SETS ───────────────────────────────────────────────────────
print("[3/5] Defining feature sets A / B / C...")

OPT15_CORE = [
    'nir', 'green', 'swir16', 'swir22', 'NDMI', 'MNDWI',
    'NDWI', 'swir_ratio', 'green_nir_ratio',
    'pet', 'ppt', 'soil', 'tmax', 'tmin',
    'temp_range', 'water_balance', 'aridity_index', 'soil_saturation',
    'month', 'season', 'day_of_year'
]
TC_PROVEN   = ['aet', 'def', 'srad']
BASE_24     = OPT15_CORE + TC_PROVEN

LAGS_A = [f'{v}_lag{l}' for v in ['ppt','soil','aet'] for l in [1,2]]      # Opt 11 proven
LAGS_B = [f'{v}_lag{l}' for v in ['ppt','soil','aet'] for l in [1,2,3]]    # + lag-3
LAGS_C = [f'{v}_lag{l}' for v in ['ppt','soil','aet','pet'] for l in [1,2,3]]  # + pet lags

def make_feats(base, lags, train_df, val_df):
    feats = [f for f in base + lags
             if f in train_df.columns and f in val_df.columns]
    return feats

FEATS_A = make_feats(BASE_24, LAGS_A, train12_lag, val12_lag)  # 30 — Opt 11 baseline
FEATS_B = make_feats(BASE_24, LAGS_B, train12_lag, val12_lag)  # 33
FEATS_C = make_feats(BASE_24, LAGS_C, train12_lag, val12_lag)  # 36

print(f"  Set A (Opt 11 baseline, lag 1-2 ppt/soil/aet) : {len(FEATS_A)} features")
print(f"  Set B (+ lag-3 ppt/soil/aet)                  : {len(FEATS_B)} features")
print(f"  Set C (+ lag 1-3 pet)                         : {len(FEATS_C)} features")

# ── PREPARE ARRAYS ────────────────────────────────────────────────────────────
print("[4/5] Preparing scaled arrays...")

def prepare_arrays(train_df, val_df, feature_list):
    X_tr = train_df[feature_list].copy()
    X_va = val_df[feature_list].copy()
    medians = X_tr.median()
    X_tr.fillna(medians, inplace=True)
    X_va.fillna(medians, inplace=True)
    sc = StandardScaler()
    return sc.fit_transform(X_tr), sc.transform(X_va)

X_trA, X_vaA = prepare_arrays(train12_lag, val12_lag, FEATS_A)
X_trB, X_vaB = prepare_arrays(train12_lag, val12_lag, FEATS_B)
X_trC, X_vaC = prepare_arrays(train12_lag, val12_lag, FEATS_C)

y_train12      = train12_lag[TARGETS].values
location_ids12 = ensure_location_id(train12_lag)['Location_ID']

# ── CV COMPARISON ─────────────────────────────────────────────────────────────
print("[5/5] Spatial CV comparison (5-fold GroupKFold)...")
print()

def run_cv(X_tr, y_arr, groups, label):
    cv_per_target = {}
    for i, tname in enumerate(['TA', 'EC', 'DRP']):
        y_t = y_arr[:, i]
        folds = []
        for tr_idx, va_idx in gkf.split(X_tr, y_t, groups=groups):
            et = ExtraTreesRegressor(**ET_PARAMS, random_state=42)
            rf = RandomForestRegressor(**RF_PARAMS, random_state=42)
            et.fit(X_tr[tr_idx], y_t[tr_idx])
            rf.fit(X_tr[tr_idx], y_t[tr_idx])
            p = ET_W * et.predict(X_tr[va_idx]) + RF_W * rf.predict(X_tr[va_idx])
            folds.append(r2_score(y_t[va_idx], p))
        cv_per_target[tname] = np.mean(folds)
        print(f"  [{label}] {tname}: {np.mean(folds):.4f}  "
              f"folds={[f'{x:.3f}' for x in folds]}")
    overall = np.mean(list(cv_per_target.values()))
    print(f"  [{label}] OVERALL: {overall:.4f}")
    print()
    return overall, cv_per_target

print("--- Set A: Opt 11 baseline (30 features) ---")
cv_A, cv_A_t = run_cv(X_trA, y_train12, location_ids12, 'A')

print("--- Set B: + lag-3 ppt/soil/aet (33 features) ---")
cv_B, cv_B_t = run_cv(X_trB, y_train12, location_ids12, 'B')

print("--- Set C: + lag 1-3 pet (36 features) ---")
cv_C, cv_C_t = run_cv(X_trC, y_train12, location_ids12, 'C')

# ── DECISION ──────────────────────────────────────────────────────────────────
print("=" * 70)
print("CV COMPARISON RESULT")
print("=" * 70)
print(f"  Set A (Opt 11 baseline) : {cv_A:.4f}  (LB ref: 0.3679)")
print(f"  Set B (+ lag-3)         : {cv_B:.4f}  delta={cv_B-cv_A:+.4f}")
print(f"  Set C (+ pet lags)      : {cv_C:.4f}  delta={cv_C-cv_A:+.4f}")
print()

THRESHOLD = 0.002
scores = {'A': (cv_A, FEATS_A, X_trA, X_vaA),
          'B': (cv_B, FEATS_B, X_trB, X_vaB),
          'C': (cv_C, FEATS_C, X_trC, X_vaC)}
best_label = max(scores, key=lambda k: scores[k][0])
best_cv, FEATS_FINAL, X_tr_FINAL, X_va_FINAL = scores[best_label]

if best_cv >= cv_A + THRESHOLD:
    print(f"  ✅ DECISION: USE SET {best_label} (CV={best_cv:.4f}, "
          f"beats A by {best_cv-cv_A:+.4f})")
    SET_LABEL = f'{best_label} ({len(FEATS_FINAL)} features)'
else:
    print(f"  ⚠ DECISION: USE SET A — no set clears +{THRESHOLD} threshold")
    print(f"     Sticking with Opt 11 feature set. Submit = reproduce 0.3679.")
    best_label, best_cv = 'A', cv_A
    FEATS_FINAL, X_tr_FINAL, X_va_FINAL = FEATS_A, X_trA, X_vaA
    SET_LABEL = f'A ({len(FEATS_A)} features — Opt 11 baseline)'

print(f"  Final: {SET_LABEL}")
print()

# ══════════════════════════════════════════════════════════════════════════════
# FULL TRAINING + SUBMISSION
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 70)
print("OPT 12: 5-Seed Ensemble + Submission")
print("=" * 70)
print(f"  Feature set  : {SET_LABEL}")
print(f"  Seeds        : 5")
print()

SEEDS = [42, 100, 200, 300, 400]

val_preds_all = []
for seed in SEEDS:
    seed_preds = np.zeros((len(X_va_FINAL), 3))
    for i in range(3):
        y_t = y_train12[:, i]
        et  = ExtraTreesRegressor(**ET_PARAMS, random_state=seed)
        rf  = RandomForestRegressor(**RF_PARAMS, random_state=seed)
        et.fit(X_tr_FINAL, y_t)
        rf.fit(X_tr_FINAL, y_t)
        seed_preds[:, i] = ET_W * et.predict(X_va_FINAL) + RF_W * rf.predict(X_va_FINAL)
    val_preds_all.append(seed_preds)
    print(f"  Seed {seed} done")

final12 = np.clip(np.mean(val_preds_all, axis=0), 0, None)

# Health check
print()
opt11_stats = {'TA': 41.91, 'EC': 161.13, 'DRP': 10.47}
for i, tname in enumerate(['TA', 'EC', 'DRP']):
    p    = final12[:, i]
    ref  = opt11_stats[tname]
    flag = "✅" if p.std() >= ref * 0.80 else "⚠ COMPRESSED"
    print(f"  {tname}: min={p.min():.1f}  max={p.max():.1f}  "
          f"mean={p.mean():.2f}  std={p.std():.2f}  (Opt11 std={ref})  {flag}")

# Save
submission12 = pd.read_csv('submission_template.csv')
for i, col in enumerate(TARGETS):
    submission12[col] = final12[:, i]
assert submission12[TARGETS].isna().sum().sum() == 0
submission12.to_csv('submission.csv', index=False)

elapsed12 = time.time() - start12
print()
print("=" * 70)
print("OPTIMIZATION 12 COMPLETE")
print("=" * 70)
print(f"  Selected set   : {SET_LABEL}")
print(f"  CV A (Opt11)   : {cv_A:.4f}  →  predicted LB ≈ {cv_A+0.090:.4f}")
print(f"  CV B (lag-3)   : {cv_B:.4f}  →  predicted LB ≈ {cv_B+0.090:.4f}")
print(f"  CV C (pet lags): {cv_C:.4f}  →  predicted LB ≈ {cv_C+0.090:.4f}")
print(f"  CV final       : {best_cv:.4f}  →  predicted LB ≈ {best_cv+0.090:.4f}")
print(f"  Elapsed        : {elapsed12:.0f}s")
print(f"  Saved          : submission.csv")
print()
print(submission12.head(6).to_string(index=False))

OPTIMIZATION 12 — Extended Lags: lag-3 + pet lags

Base  : Opt 11 (LB 0.3679) — 30 features, 5 seeds, lags 1-2
Test A: Opt 11 baseline (lag 1-2, ppt/soil/aet)  — 30 features
Test B: + lag-3 for ppt/soil/aet                 — 33 features
Test C: + lag 1-3 for pet as well                — 36 features
CV gate: must beat 0.2775 by ≥0.002 to be kept

[1/5] Engineering base features...
[2/5] Engineering temporal lags...
  Done
[3/5] Defining feature sets A / B / C...
  Set A (Opt 11 baseline, lag 1-2 ppt/soil/aet) : 30 features
  Set B (+ lag-3 ppt/soil/aet)                  : 33 features
  Set C (+ lag 1-3 pet)                         : 36 features
[4/5] Preparing scaled arrays...
[5/5] Spatial CV comparison (5-fold GroupKFold)...

--- Set A: Opt 11 baseline (30 features) ---
  [A] TA: 0.3307  folds=['0.302', '0.202', '0.363', '0.349', '0.438']
  [A] EC: 0.3677  folds=['0.326', '0.421', '0.344', '0.303', '0.445']
  [A] DRP: 0.1341  folds=['0.173', '0.138', '0.175', '0.181', '0.004']
  [A] O